# Epsilon Fund — CPCV Validation

Combinatorial Purged Cross-Validation via `infrastructure/walkforward/cpcv_engine.py`.

Run **after** walk-forward validation to get an unbiased distribution of strategy performance
across all possible train/test splits — not just one rolling path.

---
### What CPCV gives you

| Output | Meaning |
|--------|---------|
| **Path distribution** | 105 complete equity paths (N=8, k=2) — each covers the full dataset once |
| **Mean / Std Sharpe** | Is the strategy reliably positive, or just occasionally lucky? |
| **Parameter stability** | Does the optimizer converge to similar values across all 28 splits? |
| **Tercile comparison** | Do parameter choices in top-performing splits differ from poor ones? |
| **Split heatmap** | Which time periods are structurally hard for this strategy? |

---
### Workflow

1. Complete walk-forward validation for this asset first.
2. Copy `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the walk-forward notebook into the cells below.
3. Set `WF_SHARPE` to the combined OOS Sharpe from walk-forward (for comparison annotation).
4. Run all cells top-to-bottom.
5. Save the results pickle for portfolio-level analysis.

---
### Interpreting results

| Signal | Meaning | Action |
|--------|---------|--------|
| Mean path Sharpe ≈ WF Sharpe, low std | Consistent edge across all splits | High confidence — proceed |
| Mean path Sharpe positive but high std | Strategy works but period-sensitive | Review split heatmap for weak regimes |
| Many negative-Sharpe paths | Edge is concentrated in lucky folds | Revisit strategy architecture |
| CV < 0.10 across most free params | Parameters are stable, not noise-fitted | Safe to fix more params and re-run WF |
| High tercile separation on a param | That param drives performance | Consider narrowing its range or fixing it |

In [1]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from cpcv_engine import (
    run_cpcv,
    cpcv_parameter_analysis,
    cpcv_print_param_suggestions,
    cpcv_summary,
    cpcv_confidence_intervals,
    cpcv_ci_summary,
)
from cpcv_visualizer import (
    plot_cpcv_results,
    plot_parameter_distributions,
    plot_parameter_correlation_matrix,
    plot_split_performance_heatmap,
    plot_tercile_comparison,
)

---
## Data

**Use the same pair, interval, and lookback as the corresponding walk-forward notebook**
so the dataset is identical.

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

In [ ]:
SYMBOL   = 'BTCUSDT'  # ← change per asset
INTERVAL = '1d'
LOOKBACK = 2200

# ── CPCV configuration ────────────────────────────────────────────────────────
N          = 8      # groups to partition data into  →  C(8,2) = 28 splits
K          = 2      # groups held out per split
N_TRIALS   = 400    # Optuna trials per split (match walk-forward N_TRIALS)
BURNIN     = 100    # bars prepended before each test group for indicator warmup
COST       = 0.001  # per-leg trading cost (match walk-forward COST)
PURGE_BARS = 1      # training bars removed at each train/test boundary

# ── walk-forward Sharpe for comparison annotation (optional) ──────────────────
WF_SHARPE  = None   # ← set to combined OOS Sharpe from walk-forward notebook


client = get_binance_client()
df     = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-06-08 → 2026-04-27  (2150 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-04-25,77437.13,77885.35,77140.23,77625.00,5685.11420
2026-04-26,77625.00,78961.00,77326.51,78657.55,7963.56162
2026-04-27,78657.55,79485.66,76546.00,76638.00,13130.46469


---
## Strategy, PARAM_DEFS, and FIXED_PARAMS

> **Paste `strategy_fn`, `PARAM_DEFS`, and `FIXED_PARAMS` from the corresponding
> walk-forward notebook. These must match exactly.**

The CPCV engine passes the same `(df_slice, params)` interface as the walk-forward engine.
`strategy_fn` must return `(strategy_df, indicator_cols)`.  
Do not change the function signature or indicator columns — the engine relies on them for
NaN-trimming after burnin removal.

In [3]:
# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
# Only keys present in PARAMS above are searched — remove a key from PARAMS to exclude it entirely.

PARAM_DEFS = {
    'ema_span':          ('int',   5,    40),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   10,   30),
    'atr_size':          ('int',   3,    14),
    'adx_override':      ('int',   52,   65),
    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_caution': ('float', 0.1, 0.6),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 1.0, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
    'obv_ma_period':  ('int',   10,  40),   # OBV smoothing window
    'obv_lookback':   ('int',   10,  30),   # bars back to compare price vs OBV direction
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = {
    'risk_per_trade': 0.0426,
    'max_leverage': 2.8325,

    'stop_atr_scale': 1,
    'stop_mult_pos_normal':1,
    'stop_mult_ent_normal': 1,

#    'obv_ma_period': 32,
#    'obv_lookback': 16,
#    'adx_override': 60,
    }

In [4]:
# ── strategy function ─────────────────────────────────────────────────────────
# Paste from walk-forward notebook — must be identical.
# Required signature: my_strategy(df_slice: pd.DataFrame, params: dict)
# Required return:    (strategy_df, indicator_cols)

def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()

    # ── indicators ────────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=params['ema_span'], adjust=False).mean()
    df['Swing_Hi_Cau'] = df['High'].rolling(params['swing_caution']).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(params['swing_caution']).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(params['swing_stop']).max()

    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(params['atr_caution'])
    df['ATR_Stp'] = atr(params['atr_stop'])
    df['ATR_Sz']  = atr(params['atr_size'])

    up    = df['High'].diff();  down = -df['Low'].diff()
    pdm   = up.where((up > down) & (up > 0), 0.0)
    ndm   = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(14)
    pdi   = 100 * pdm.ewm(span=14, adjust=False).mean() / atr14
    ndi   = 100 * ndm.ewm(span=14, adjust=False).mean() / atr14
    dx    = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=14, adjust=False).mean()

    #df['Vol_MA']  = df['Volume'].rolling(params['vol_ma_period']).mean()
    direction     = df['Close'].diff().apply(lambda x: 1 if x > 0 else -1)
    df['OBV']     = (df['Volume'] * direction).cumsum()
    df['OBV_MA']  = df['OBV'].rolling(params['obv_ma_period']).mean()

    df['Caution_OBV']   = (df['Close'] > df['Close'].shift(params['obv_lookback'])) & (df['OBV'] < df['OBV_MA'])
    df['Caution_Long']  = ((df['Swing_Hi_Cau'] - df['Low']) > 1.5 * df['ATR_Cau']) | df['Caution_OBV']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > 1.5 * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    _valid = df['Swing_Hi_Stp'].notna() & df['ATR_Stp'].notna() & df['ATR_Sz'].notna() & df['OBV_MA'].notna() #& df['Vol_MA'].notna()
    df['Entry_Long']    = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > params['adx_override'])) & _valid # & (df['Volume'] > df['Vol_MA']) 
    df['position_size_raw'] = (params['risk_per_trade'] / (df['ATR_Sz'] / df['Close'])).clip(0.1, params['max_leverage'])

    n             = len(df)
    position      = [0]      * n
    position_size = [0.0]    * n
    stop_arr      = [np.nan] * n
    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        # 1. First, check if we need to exit an existing position
        if in_position == 1:
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                # If no exit, update the trailing stop as usual
                sm        = params['stop_mult_pos_caution'] if curr['Caution_Long'] else params['stop_mult_pos_normal']
                stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale'])

        # 2. Second, check if we should enter (even if we just exited above!)
        if in_position == 0:
            if curr['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']
                cl = curr['Caution_Long']; cs = curr['Caution_Short']
                if cl and cs: sm = params['stop_mult_ent_both']
                elif cl:      sm = params['stop_mult_ent_caution']
                else:         sm = params['stop_mult_ent_normal']
                stop_loss = curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale']
        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau', 'OBV_MA']
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss']     = df['stop_loss'].fillna(0.0)

    return df, indicator_cols


---
## Run CPCV

Partitions the data into `N` equal groups and optimises on every `C(N, K)` training
complement. Each of the 28 splits produces an OOS evaluation on the `K` held-out groups.
All groups are then stitched into 105 complete equity paths (for N=8, k=2).

| Parameter | Description | Default |
|-----------|-------------|---------|
| `N` | Groups — 8 gives C(8,2)=28 splits and 105 complete paths | `8` |
| `K` | Groups held out per split | `2` |
| `N_TRIALS` | Optuna trials per split — match walk-forward for consistency | `400` |
| `BURNIN` | Bars prepended to each test group for indicator warmup | `100` |
| `PURGE_BARS` | Training bars purged at each train/test boundary | `1` |
| `COST` | Per-leg trading cost fraction | `0.001` |

> **Runtime:** `C(N, K) × N_TRIALS` Optuna evaluations.  
> For N=8, K=2, N_TRIALS=400: ~11,200 backtests — expect 10–30 min on daily data.

In [ ]:
# ── scoring function ─────────────────────────────────────────────────────────
# Weights each metric component before Optuna maximises the score.
# Tune the MAX caps and weights to emphasise what matters for this strategy.
# Set SCORE_FN = None to fall back to the engine default (same weights).

def SCORE_FN(metrics):
    SHARPE_MAX = 2.5
    CALMAR_MAX = 60.0
    RETURN_MAX = 15.0
    calmar = (metrics['total_return'] / abs(metrics['max_drawdown'])
              if metrics['max_drawdown'] != 0 else 0.0)
    s = np.clip(metrics['sharpe_ratio'] / SHARPE_MAX, 0, 1)
    c = np.clip(calmar                  / CALMAR_MAX, 0, 1)
    r = np.clip(metrics['total_return'] / RETURN_MAX, 0, 1)
    return 0.50 * s + 0.30 * c + 0.20 * r

# ── rejection filter ─────────────────────────────────────────────────────────
# Trials that return True are scored -999 and discarded by Optuna.
# Set REJECT_FN = None to use the engine default.

def REJECT_FN(metrics):
    if metrics is None:               return True
    if metrics['num_trades']   < 7:  return True
    if metrics['win_rate']     < 0.35: return True
    if metrics['max_drawdown'] < -0.80: return True
    if metrics['profit_factor'] < 0.8:  return True
    return False

In [ ]:
results = run_cpcv(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    N            = N,
    k            = K,
    purge_bars   = PURGE_BARS,
    n_trials     = N_TRIALS,
    burnin       = BURNIN,
    cost         = COST,
    score_fn     = SCORE_FN,     # ← customise above; set None for engine default
    reject_fn    = REJECT_FN,    # ← customise above; set None for engine default
    verbose      = True,
)

---
## Text Summary

Prints group date ranges, path distribution percentiles, and the top/bottom 5 paths
by Sharpe with a split legend.

### Reading the notation

**Groups** (`g0`–`g7`) are contiguous time segments that tile the full dataset.  
Their date ranges are printed at the top of the summary output.

**Splits** (`s0`–`s27`) are the 28 = C(8,2) optimisation runs.  
Each split holds out exactly **2 groups** for OOS evaluation and **trains on the remaining 6**.
The split number is just an index — see the legend below the top/bottom table to decode which
groups each split tested and which it trained on.

**Paths** stitch together 4 splits whose test groups cover all 8 groups exactly once,
producing one continuous equity curve from the start of g0 to the end of g7.

**`g3→s11`** means: *"group 3's OOS data in this path came from split 11's evaluation"*.  
Split 11 held out g3 (and one other group) and was optimised on the remaining 6.
The split legend at the bottom of the summary shows — for every sN in the highlights —  
exactly which groups it tested, the test date range, and which groups it trained on.

Pass `show_paths=True` to also print the full per-path table (105 rows).

In [ ]:
# Group legend + path distribution percentile table
cpcv_summary(results, show_highlights=False, show_split_legend=False, show_ci=False)
# add show_paths=True to also print the full 105-row per-path table

In [ ]:
# Top/bottom highlights + split legend + confidence intervals
cpcv_summary(results, show_group_legend=False, show_distribution=False)

---
## Path-Level Charts

Three outputs covering the full path distribution:

- **Equity curves** — all 105 paths (semi-transparent blue), the mean path (bold amber),
  and the min–max envelope (shaded). Vertical dotted lines mark group boundaries g0–g7,
  so you can see which time segments contributed most variance.
- **Sharpe histogram** — distribution of the 105 path-level Sharpe ratios with an
  overlap-adjusted confidence interval overlay. A tight, right-skewed histogram means
  the edge is consistent; a wide or left-skewed one means some time periods badly hurt
  the strategy.
- **Confidence interval text table** — printed before the charts. Shows the naive
  (anticonservative) and overlap-adjusted (conservative) CIs for Sharpe, Calmar, and
  Max DD, plus the conservative floor on the Sharpe.

### Confidence interval overlay
The histogram draws a **blue shaded band** for the overlap-adjusted CI and a **dashed
red line** at the conservative lower bound. The adjusted CI uses an effective sample size
`N_eff = N² / sum(overlap_matrix)` — because CPCV paths share splits, the 105 paths are
not independent, so `N_eff` is substantially smaller than 105. The conservative lower
bound is the most cautious point estimate of mean Sharpe; use it when comparing against
a hurdle rate.

In [ ]:
ci = cpcv_confidence_intervals(results)
plot_cpcv_results(results, wf_sharpe=WF_SHARPE, ci_results=ci);

---
## Parameter Analysis

Compute the analysis object once — all four charts below read from it.

In [8]:
analysis = cpcv_parameter_analysis(results)

# Prints consensus_ranges table + copy-pasteable PARAM_DEFS / FIXED_PARAMS blocks.
# CV < 0.10  → "fix at {median}"   : converges consistently, safe to anchor.
# CV 0.10–0.25 → "narrow to IQR"  : reduce search range to Q25–Q75 next WF run.
# CV > 0.25  → "keep current range": period-sensitive, keep free.
cpcv_print_param_suggestions(results, analysis)

                       recommended_low  recommended_high  current_low  current_high              action
param                                                                                                  
ema_span                     17.750000         29.500000          5.0          40.0  keep current range
swing_caution                 3.000000         10.000000          3.0          14.0  keep current range
swing_stop                    5.750000          8.250000          3.0          10.0  keep current range
atr_caution                  13.000000         29.000000         10.0          30.0  keep current range
atr_stop                     11.000000         23.250000         10.0          30.0  keep current range
atr_size                      5.000000         11.000000          3.0          14.0  keep current range
adx_override                 53.000000         59.500000         52.0          65.0           fix at 54
stop_mult_pos_caution         0.276495          0.526004        

### Parameter Distributions

One subplot per free parameter. Each dot is one split's best value, jittered horizontally
for readability and **coloured by that split's OOS Sharpe** (red = low, green = high).

- **IQR band** (shaded blue) — the middle 50% of optimised values across splits.  
  A narrow band means the optimizer consistently lands in the same region (stable).
- **Median line** (amber dashed) — the central tendency; this is the value suggested
  by `fix at {median}` if you decide to anchor the parameter.
- **Subplot subtitle** — shows the CV verdict (Stable / Moderate / Scattered) and
  the recommended action from `consensus_ranges`.

**What to look for:** Dots that are clustered tightly (narrow IQR, low spread) and
coloured consistently green indicate a parameter the strategy genuinely relies on at
a stable value — a good candidate to fix. Dots scattered randomly with mixed colours
indicate noise-fitting: the optimizer fills in a random value each time because the
parameter doesn't matter, or is too regime-dependent to converge.

In [9]:
plot_parameter_distributions(results, analysis=analysis);

### Parameter Correlation Matrix

Pearson correlation between every pair of free parameters **across the 28 splits**.
Each cell value is how consistently the two parameters move together when Optuna
optimises on different training windows.

- **Red** (r close to −1) — strong negative correlation: when the optimizer raises one
  parameter it tends to lower the other. The two may be partially substitutable.
- **Blue** (r close to +1) — strong positive correlation: both parameters rise or fall
  together. One might be redundant — consider fixing the less interpretable one.
- **White** (r ≈ 0) — independent. Each parameter is doing its own job.

Only the lower triangle is shown; the upper is the mirror image.

**What to look for:** Any |r| > 0.6 pair is worth investigating. High correlation can
mean the search space has a ridge the optimizer slides along freely — removing one
parameter or fixing the ratio between them may reduce overfitting.

In [10]:
plot_parameter_correlation_matrix(analysis);

### Split Performance Heatmap

An N × N grid where cell (i, j) shows the **OOS Sharpe of the split that held out
groups i and j** for testing (trained on the other 6 groups).

- **Green** cells — that group-pair combination was OOS-profitable. The strategy
  generalises well when those two periods are unseen.
- **Red** cells — those two periods were hard OOS. Either the strategy has no edge
  in those regimes, or the training data available (6 groups) wasn't representative.
- **Blank** cells — diagonal and lower triangle (redundant; each split appears once).

**What to look for:** A consistent block of red cells around a particular group index
means one time period is structurally difficult for this strategy regardless of which
other period it's paired with. Cross-reference the group date legend in `cpcv_summary`
to identify that regime (e.g. a bear market or low-volatility period).

In [11]:
plot_split_performance_heatmap(results);

### Tercile Comparison

One subplot per free parameter. Each subplot shows **two box plots side by side**:

- **Green box** — parameter values chosen by the top ⌊N/3⌋ splits ranked by OOS Sharpe.
- **Red box** — parameter values chosen by the bottom ⌊N/3⌋ splits.

The **separation score** in the subtitle is `|mean_top − mean_bottom| / pooled_std`.  
It measures how far apart the two distributions are relative to their spread.

| Separation | Interpretation |
|------------|----------------|
| < 0.5 | No meaningful difference — parameter choice doesn't predict split quality |
| 0.5–1.0 | Moderate signal — consider narrowing the range toward the top-tercile median |
| > 1.0 | Strong signal — this parameter's value is predictive of OOS performance |

**What to look for:** Parameters where the green and red boxes barely overlap and the
separation is > 1 are the most actionable — the optimizer is reliably choosing higher
(or lower) values in splits that perform well. Consider anchoring to the top-tercile
median or narrowing the search range to that region.

In [12]:
plot_tercile_comparison(results, analysis);

---
## Save Results

Pickle the full results dict for portfolio-level analysis and future reference.

In [13]:
results_file = f'{SYMBOL.lower()}_cpcv.pkl'
pd.to_pickle(results, results_file)
print(f'Saved to {results_file}')

Saved to btcusdt_cpcv.pkl
